In [1]:
%run "../../Latex_macros.ipynb"

<IPython.core.display.Latex object>

<html>
<p style="font-size:32px"><strong>Final Project, Part 2: Post-training with Reinforcement Fine-tuning</strong></p>
</html>

# Assignment

*Pre-training* an LLM involves training the model to solve the Language Modeling task (predict the next token)
- "PT" in "GPT" stands for Pre Trained

Supervised Fine Tuning can adapt the model to other tasks
- this was the subject of Part 1 of the Final Project

Other useful behaviors and skills are imparted to the model by Post Training, often using Reinforcement Learning
- Reinforcement Fine Tuning (RFT)

The objective of this part of the Final Project is for you to demonstrate a depth of knowledge of the topic.

In this part of the Final Project, you will use RFT on a Pre-Trained model in order to
train it on a new skill: using a Quant Library.


## Project design

Coding projects in the Coding Assistant era become a challenge as a means of differentiating the depth of
knowledge that a student has acquired.  The project design is an attempt to measure the depth of your knowledge.

I fully expect you to use an AI Assistant/Agent to complete the project.
My experience is that doing so will not present much of an intellectual challenge to you:
the result is high quality code

But everyone else's code will be high quality too. You will need to find a way to distinguish your work from
everyone else's.

The tasks that you are asked to perform are the
*minimum* that you are required to perform and will account for $\frac{2}{3}$ of the total points
that are possible to earn for this part of the Project.

The other $\frac{1}{3}$ of the total points that are possible will be at the grader's discretion.
It will be based on
- your performance of tasks of your own choice that are *in addition* to the *minimum*
    - the minimum submission is just that, a minimum
    - please consider: what else can I do to demonstrate deep knowledge
        - propose a new task, justify why it is interesting, perform it, evaluate/compare to the other tasks

Additionally: the instructions in this notebook are intentionally minimal (to provide less detail to your coding assistant).
Please ask questions in person, in class and to the TA, if anything is not clear.


## Format of your submission

Your submission will be a *well structured* Jupyter notebook that is clearly presented.
- Follow the Recipe for Machine Learning that we discuss in the FRE 7773 course
- The TA should be able to open the notebook, press "Run all cells" and have the notebook execute without error to the end
- The TA should be able to *easily* find
    - your results for each required task
    - the answer to any question that is posed as part of the task
    - your unique "above and beyond" contribution
        - clearly explain what you are doing
        - clearly explain why this is interesting
        - perform your experiment
        - analyze the results and report any insights/things that you learned
        - even a "failed" experiment can be interesting if you learned or discovered something

Since there are multiple parts: please use Section headings to assist the TA in identifying each part.


## Requirements

You should use HuggingFace tools to complete this assignment.

The use of PyTorch (rather than Keras) will be necessary
- due to the limited support for Keras in HuggingFace

The use of a GPU is highly desirable.


You are to fine tune a *base model*
- to call functions in a Quant Library
- *when appropriate* (i.e., for prompts that are best answered by tools in the library)

**Details**
- The *base model* that you will tune is `'HuggingFaceTB/SmolLM2-135M-Instruct'`
- Use **GRPO** as the RL algorithm for the Reinforcement Learning phase.
    - The standard RFT recipe has **two phases**: a Supervised Fine-Tuning (SFT) part, followed
      by an RL part — in this assignment the RL algorithm is GRPO. SFT is *not* itself part of
      GRPO; it is a separate supervised step that produces the reference checkpoint GRPO then
      improves.
    - There may be restrictions for each part: please read details carefully and obey these restrictions

- The *minimum* Quant Library has one tool: a Black-Scholes option pricing calculator.


This tool is implemented by a function named `bs` with the following signature:

```
def bs(S: float, K: float, T: float, r: float, sigma: float, type: str) -> float:
    """Calculates the theoretical price of a European option using the Black-Scholes model.

    Args:
        S: The current spot price of the underlying asset.
        K: The strike (exercise) price of the option.
        T: The time to maturity expressed in years.
        r: The annualized risk-free interest rate as a decimal (e.g., 0.05 for 5%).
        sigma: The annualized volatility of the underlying asset as a decimal
          (e.g., 0.16 for 16%).
        type: The type of option to price. Must be either "call" or "put".

    Returns:
        The theoretical fair market value (premium) of the option.

    Examples:
        >>> bs(S=100, K=110, T=.5, r=5.1/100, sigma=16/100, type="call")
        1.92
    """
```


When the model wants to call a tool in response to a prompt
- it outputs *text* in the following form

```
<tool_code>bs(S=100, K=110, T=.5, r=5.1/100, sigma=16/100, type="call")</tool_code>
```

That is
- there is a call to a function (e.g., `bs`)
- with the necessary arguments
- bracketed by `<tool_code>` delimiters


The desired behavior of your fine-tuned model is:
- when presented with a prompt
- that is best answered by calling a tool in the library
- the response should be a call to the appropriate function, as illustrated above


**Simplifying assumptions**
- You **don't need to implement** any function in the Library
- you **don't need to actually call** the function

You only need to generate a response that is syntactically correct according to the above.

This avoids the need for you to have to deal with Python details of calling arbitrary functions that
are only represented as strings.


**If we were to use this fine-tuned model in the real world**
- responses from the model would be intercepted by a "supervisor" (a "harness")
- which would recognize strings with the `<tool_code>` delimiters
- and call the function
- *replacing* the original response
- with the result of the function call

From the user's perspective: they get just the answer without seeing the function call string.

**We are NOT** going to implement the harness; this is just FYI.


# Tasks

**Requirements**

- You are asked to report multiple aspects of your process
    - Make it easy for the Grader to identify what you are responding to and what the response is.
    - Give your thought process about the purpose of the question and the result you report
- The major goal is to **add** a skill (Quant Library usage) to an already capable model
    - You **should not** create a system prompt (if you decide to use one) that is overly-specific to the goal of using a Quant Library
        - the model should learn to use the library via GRPO, not instructions in the prompt


## Structure of the assignment

The assignment is organized into **two required layers** plus an **above-and-beyond** layer.

| Layer | Part(s) | Data | Starting model |
|:-------|:---------|:------|:---------------------|
| **Layer 1** — main RFT task | SFT phase **+** GRPO phase | Standardized units | Base model |
| **Layer 2** — challenge task | SFT phase **+** GRPO phase | Standardized and Non-standardized units | Base model |
| **Above & Beyond** (extra credit) | Your choice | Your choice | Your choice |

The two required layers are the "minimum"; the extra-credit layer is where you distinguish your
submission from everyone else's.


## Generate training and evaluation examples

You need to create prompts in **natural language** (what a human user would do).

For example
```
What is the price of an option with 0.5 years to expiration and strike $ 110 when the current underlying
is priced at $ 100, the discount rate is .051 and the volatility is .16
```

Human users could ask this same question in diverse ways.

Your examples should represent this diversity. Some ideas:
- phrase the question in different ways. 
    - "What is the price of an option ...."
    - "Given an option with .... what is the price ?"
- use different names for attributes: Strike/exercise price; maturity/expiry; vol/volatility
- Present attributes in different orders




**Requirement**

- Indicate the size of each dataset you create
- Indicate the *number of formats* you use for examples
- Demonstrate the diversity of your examples:
    - Use *at least 5* formats for the prompt; give them each a name
        - e.g., "Type 1", "Type 2", etc., or "Direct", "Informal", etc.
    - Choose 3 examples of each type from your dataset and print them out
    - Create a histogram of the example distribution
        - of *fraction* of training set per type
- We want to report how close each phase has gotten us to our goal
    - Create *at least one* Performance Metric
        - must include a metric "Average Reward"
    - Clearly indicate the value of the metrics achieved for each phase
    
For **Layer 1**: 

As a simplification: 
your prompts may use attribute values of the exact type according to the function signature.

These **standardized units**
- rate as decimal like `0.051`
- time  in years like `0.5`,
- volatility as decimal like `0.16` 

For **Layer 2**: 

We increase the difficulty by changing the datasets so that 
 the units in the prompt are *not restricted* to Standardized units.


# Layer 1 — main RFT task (SFT + RL, starting from the base model)

Layer 1 fine tunes the model on training examples where prompts are in **standardized units**.

The goal of the fine-tuned model is to output
the correct function call, bracketed by the  `<tool_code>` delimiters as indicated above.

Here is a sample example:

PROMPT:
What is the Black-Scholes price of a put with S=80, K=75, 1 year to expiry, r=.025, sigma=.18?

RESPONSE:
`<tool_code>bs(S=80, K=75, T=1, r=2.5/100, sigma=18/100, type="put")</tool_code>`

## Training — Layer 1

Layer 1 has **two phases**: an **SFT phase** and a **GRPO phase**. 

- SFT comes first and produces
the *SFT-trained model* (reference model)
- GRPO applied Reinforcement Learning to the SFT-trained model and produces the *GRPO Layer 1-trained model*

GRPO  requires you to create a *Reward Function*

**Requirements (Layer 1)**

- Your reward for a generated response should be in the range $[0,1]$
- Describe the Reward function in detail
    - how and why you assigned "sub-rewards" for specific parts of the response
- Choose 3 prompts of each type.  For each prompt
    - print out
        - the prompt type
        - the prompt
        - one response
        - the reward for the response
- We want to report how close each phase has gotten us to our goal
    - Create *at least one* Performance Metric
        - must include a metric "Average Reward"
    - Clearly indicate the value of the metrics achieved for **each stage** (base → after SFT → after GRPO)


### SFT part — Layer 1

Depending on
- the diversity of your examples
- the size of your SFT dataset
- the number of training steps

it may be possible that the SFT phase completely or mostly solves the RFT task, with little or no need for
the GRPO phase.

To ensure that the GRPO phase is a substantial challenge, we will restrict the SFT training
- based on the *cumulative number* of examples processed

Suppose you train for $s$ steps, with batch size $b$ in each step
- The *cumulative number of examples processed* is $s \times b$
- This is **not** the number of *unique* examples
    - If you train for $e > 1$ epochs, examples will repeat

**Requirement**
- the cumulative number of examples processed in the Layer 1 SFT phase
    - should not exceed **1000**
    - a reference configuration that fits the cap: `max_steps=50`, `batch_size=16`, `grad_accum=1`
      → cumulative $50 \times 16 = 800$ examples
- Indicate the cumulative number of examples processed in the SFT part


### GRPO phase — Layer 1

**Requirements (Layer 1 GRPO)**

- Indicate the group size $G$ that you use
- Indicate the settings of any other parameters that you consider significant
    - in particular the KL-to-reference coefficient $\beta$ and the RL learning rate


# Layer 2 — Challenge task (SFT + RL, starting from the base model, without standardized units simplification)

The difficulty of the GRPO part may decrease if the SFT part was too successful.

To test your skill at the GRPO part, we give you an additional *Challenge Task*.

This task is identical to Layer 1, but with a different dataset
- we remove the standardized units simplification
- you will perform new SFT and RL phases
- obeying the **same** goals, requirements and restrictions as Layer 1

**Setup**

- Training with SFT + RL will result in the *GRPO Layer 2-trained model*
- **You will need to create new datasets for Layer 2**
    - The Layer 2 dataset is identical in spirit to Layer 1, but we remove the restriction that examples use only standardized units
    - rate may be given either in decimal ('.051') or '%' (`5.1 %` or `5.1 percent`)
    - volatility may be given either in decimal (.16) or `%` (`16 %` or `16 percent`)
    - time may appear in **days**, **months**, *or* **years**
        - assume 365 days/year; 12 months/year

The model may need to perform 
*unit conversions* (from the given units to the units specified in the signature for the `bs` function)
on the parameter values.
  
The *recommended*  way to create the unit conversion is division/multiplication by a conversion factor 

Some examples
  - "Rate is 5 %" should result in argument `r=5/100`
  - "with 16% volatility" should result in argument `sigma=16/100`
  - "expiring in 90 days" should result in argument `T=90/365`

- The full unit grid is $3 \times 3 \times 3 = 27$ combinations, sampled across all $\geq 5$ prompt
  styles from Layer 1.

Here is a sample example:

PROMPT:
Hey,  can you price a put option, spot 120 strike 115 maturity 3 months rate 4 % vol 22% ? 

RESPONSE:
`<tool_code>bs(S=120, K=115, T=3/12, r=4/100, sigma=22/100, type="put")</tool_code>`



## Requirements -- Layer 2

In *addition* to the requirements for Layer 1:
- report Average Reward, based on the Layer 2 validation dataset, for the GRPO Layer 1-trained model 
    - this measures how different the GRPO Layer 2-trained model is from the GRPO Layer 1-trained model
      


# Above and Beyond (extra credit)

The minimum submission stops at Layer 2. 

If you choose, you may submit additional contributions beyond what was asked.
The purpose is for you to demonstrate *deep* knowledge of RFT-tuning.

You are free to propose anything; we will use our discretion in assessing the difficulty/value of your contribution.


---

## A suggestion to consider

Your GRPO Layer 2-trained  model is very good at emitting `<tool_code>bs(...)</tool_code>` for option-pricing
prompts. 

But remember: the base model was quite capable at other tasks.

Question: your GRPO layer 2-trained model has learned a new task (option pricing via a quant library);
is it as capable on other tasks as it was before being fine tuned for the new task ?

**Requirements (if you attempt this)**
- clearly describe the phenomenon you observed and why it is interesting
- *quantitatively* measure  the phenomenon
- propose, justify *and implement* at least one mitigation strategy, if you feel one is needed
    - report on the success/failure of the strategy
    - suggest further followups


In [2]:
print("Done")

Done
